In [1]:

!pip -q install datasets transformers evaluate scikit-learn torch accelerate matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [1]:
import os
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "TRUE"
os.environ["WANDB_DISABLED"] = "true"

import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
from datasets import load_dataset

dataset = load_dataset("glue", "qqp")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}


In [3]:
train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).rename("proportion"))
print(train_df["label"].value_counts().rename("count"))

train_df["q1_len"] = train_df["question1"].fillna("").str.split().str.len()
train_df["q2_len"] = train_df["question2"].fillna("").str.split().str.len()
print("\nQuestion length summary:")
print(train_df[["q1_len", "q2_len"]].describe())

Train size: 363846
Validation size: 40430

Train label distribution:
label
0    0.630673
1    0.369327
Name: proportion, dtype: float64
label
0    229468
1    134378
Name: count, dtype: int64

Question length summary:
              q1_len         q2_len
count  363846.000000  363846.000000
mean       10.942822      11.183468
std         5.437638       6.325961
min         1.000000       1.000000
25%         7.000000       7.000000
50%        10.000000      10.000000
75%        13.000000      13.000000
max       125.000000     237.000000


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def pair_to_text(df):
    return (df["question1"].fillna("") + " [SEP] " + df["question2"].fillna("")).tolist()

# Use a manageable subset for faster baseline training. Increase if you have time.
BASELINE_TRAIN_SIZE = 50000
baseline_train = train_df.sample(n=min(BASELINE_TRAIN_SIZE, len(train_df)), random_state=SEED)

X_train_base = pair_to_text(baseline_train)
y_train_base = baseline_train["label"].values
X_val_base = pair_to_text(val_df)
y_val = val_df["label"].values

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=100000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

baseline.fit(X_train_base, y_train_base)
baseline_preds = baseline.predict(X_val_base)

print("TF-IDF baseline results:")
print(classification_report(y_val, baseline_preds, digits=4))
print("Confusion matrix:", confusion_matrix(y_val, baseline_preds))

TF-IDF baseline results:
              precision    recall  f1-score   support

           0     0.8058    0.7932    0.7994     25545
           1     0.6544    0.6720    0.6630     14885

    accuracy                         0.7486     40430
   macro avg     0.7301    0.7326    0.7312     40430
weighted avg     0.7501    0.7486    0.7492     40430

Confusion matrix: [[20262  5283]
 [ 4883 10002]]


In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["question1"],
        batch["question2"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["question1", "question2", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 390965
    })
})


In [7]:
TRAIN_SIZE = 30000
EVAL_SIZE = 5000

train_data = tokenized_dataset["train"].shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(tokenized_dataset["train"]))))
eval_data = tokenized_dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(tokenized_dataset["validation"]))))

print(train_data)
print(eval_data)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./bert-qqp-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.368813,0.338144,0.853000,0.770571,0.859147,0.812452
2,0.237563,0.375870,0.859400,0.792175,0.841338,0.816017
3,0.153764,0.533027,0.859400,0.792175,0.841338,0.816017


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5625, training_loss=0.27466019965277777, metrics={'train_runtime': 2289.1768, 'train_samples_per_second': 39.315, 'train_steps_per_second': 2.457, 'total_flos': 5919998745600000.0, 'train_loss': 0.27466019965277777, 'epoch': 3.0})

In [10]:
bert_metrics = trainer.evaluate()
print(bert_metrics)

{'eval_loss': 0.37714725732803345, 'eval_accuracy': 0.859, 'eval_precision': 0.7916666666666666, 'eval_recall': 0.8407987048030221, 'eval_f1': 0.8154933263543575, 'eval_runtime': 40.7185, 'eval_samples_per_second': 122.794, 'eval_steps_per_second': 3.856, 'epoch': 3.0}


In [11]:
pred_output = trainer.predict(eval_data)
logits = pred_output.predictions
labels = pred_output.label_ids
preds = np.argmax(logits, axis=-1)

print(classification_report(labels, preds, target_names=["not_paraphrase", "paraphrase"], digits=4))
print("Confusion matrix:", confusion_matrix(labels, preds))

                precision    recall  f1-score   support

not_paraphrase     0.9027    0.8697    0.8859      3147
    paraphrase     0.7917    0.8408    0.8155      1853

      accuracy                         0.8590      5000
     macro avg     0.8472    0.8553    0.8507      5000
  weighted avg     0.8616    0.8590    0.8598      5000

Confusion matrix: [[2737  410]
 [ 295 1558]]


In [12]:
from scipy.special import softmax

probs = softmax(logits, axis=1)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for threshold in thresholds:
    tuned_preds = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, tuned_preds, average="binary", zero_division=0)
    rows.append({"threshold": threshold, "precision": precision, "recall": recall, "f1": f1})

threshold_df = pd.DataFrame(rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

best_threshold = threshold_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
print("Best threshold:", best_threshold)

,threshold,precision,recall,f1
9,0.55,0.799587,0.835402,0.817102
4,0.30,0.766430,0.874798,0.817036
3,0.25,0.761062,0.881813,0.817000
2,0.20,0.751815,0.894226,0.816860
5,0.35,0.773166,0.864544,0.816306
8,0.50,0.791667,0.840799,0.815493
7,0.45,0.785107,0.847814,0.815257
6,0.40,0.777451,0.855909,0.814796
1,0.15,0.742997,0.901781,0.814725
10,0.60,0.803468,0.825148,0.814164


Best threshold: 0.5500000000000002


In [14]:
# Build a readable validation dataframe aligned with eval_data selection.
eval_size = len(eval_data)
raw_eval_sample = dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(dataset["validation"]))))
errors_df = pd.DataFrame(raw_eval_sample)
errors_df["pred"] = preds
errors_df["paraphrase_probability"] = probs
errors_df["error_type"] = np.where(
    (errors_df["label"] == 0) & (errors_df["pred"] == 1), "false_positive",
    np.where((errors_df["label"] == 1) & (errors_df["pred"] == 0), "false_negative", "correct")
)

print("False positives:")
display(errors_df[errors_df["error_type"] == "false_positive"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))

print("False negatives:")
display(errors_df[errors_df["error_type"] == "false_negative"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))

False positives:


,question1,question2,label,pred,paraphrase_probability
0,How do I get rid of anxiety and depression?,"How do I get rid of anxiety, loneliness and de...",0,1,0.858367
21,How do I make dreams come true successfully?,How can I make my dreams come true?,0,1,0.930090
38,Where is the strangest place you've ever mastu...,Where is the weirdest place you've had sex?,0,1,0.928771
52,What are the best new Car technology that most...,What are the best new car products or inventio...,0,1,0.918342
62,How much money can I withdraw through self che...,How much money can I withdraw through cheque?,0,1,0.762898
64,What type of guys do girls like?,What type of girls do guys like?,0,1,0.871894
85,What is the total number of hospitals in India...,"How many hospitals do we have in India, includ...",0,1,0.860114
87,What is the best iphone keyboard?,What is the best iPhone?,0,1,0.965092
100,How can I prepare for English on the CAT?,Will my English become better during preparati...,0,1,0.831996
108,How many days late from your period should pas...,How long should you wait before seeing a docto...,0,1,0.837709


False negatives:


,question1,question2,label,pred,paraphrase_probability
36,What are the pros and cons of buying a used ca...,Is buying used cars from Avis or Hertz a good ...,1,0,0.365335
53,What do you think of the new iPhone7 Airpods?,What do you think of Apple's airpods?,1,0,0.022976
94,What is the stock market?,What is sensex all about and how to understand...,1,0,0.086024
107,What are the best places to visit near Bangalo...,What are some good places to visit near Bengal...,1,0,0.030970
117,What is the point of living when were all goin...,What is the point of living if we are going to...,1,0,0.028653
148,Are/were mermaids/mermen real?,Are maremaids real?,1,0,0.000548
204,Is Hillary Clinton's enabling of voter fraud a...,What is your opinion on the O'Keefe video expo...,1,0,0.003195
229,How much does a sheet of paper weigh?,How much does 8 sheets of paper weigh?,1,0,0.046489
233,How do you set up voicemail on an iPhone?,How can I active voicemail alerts on my iPhone?,1,0,0.497150
240,How do I post a question in quora?,How do I post here?,1,0,0.309469
